# 1b · Train an anomaly model

**Core · Live-code · ~25 min**

In **1a** you spotted incidents with your eyes. That doesn't scale — the crew can't watch
charts 24/7. So now we'll build a small **machine-learning model** that flags unusual
readings automatically.

The key idea we're about to use is called **unsupervised anomaly detection**. Let's unpack
those words before we code.

## The concept: learning "normal" without being told the answers

- **Supervised** learning means you train a model on labelled examples ("this is an incident,
  this is not"). It needs an answer key for training.
- **Unsupervised** learning means the model gets **no labels** — it just studies the data and
  figures out structure on its own.

**Anomaly detection** is the task of finding the points that don't fit the pattern. We'll use
an unsupervised method called **Isolation Forest**. Here's the intuition:

> Imagine playing "20 questions" to isolate one data point from all the others. A totally
> normal reading (near the middle of the pack) takes *many* questions to pin down, because
> lots of readings look like it. A weird outlier takes *very few* questions, because it's
> already off on its own. Isolation Forest scores each point by how *easily* it can be
> isolated — easy-to-isolate points are flagged as anomalies.

Crucially, we will **not** show the model our `label` column during training. We hold that
answer key back and only use it *afterwards* to grade the model — just like real life, where
you usually don't know the answers in advance.

## Step 1 — Load the data and choose the features

A **feature** is simply an input column the model learns from. We give the model the seven
sensor signals. We deliberately *exclude* `timestamp`, `module` (text, not a measurement),
and `label` (that's the answer key — feeding it in would be cheating).

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

# Reload and clean in one line (same steps as notebook 1a).
df = pd.read_csv("../data/telemetry.csv", parse_dates=["timestamp"]).ffill().bfill()

# The seven sensor signals the model will learn from.
features = ["habitat_temp_c", "o2_pct", "co2_ppm", "power_kw", "battery_pct", "dust_index", "water_l"]
X = df[features]   # capital X is the conventional name for "the input features"
X.head()

## Step 2 — Train the detector

We create an `IsolationForest` and let it study `X`.

- `contamination=0.1` is our rough guess that about **10%** of readings are anomalous. (Our
  injected incidents are roughly that share of the week.) This tells the model how
  trigger-happy to be.
- `random_state=42` makes the result **reproducible** — the algorithm uses randomness, and
  fixing the seed means you and your neighbour get the same answer.

`fit_predict(X)` does two things at once: it *learns* from the data and *labels* each row.
It returns **-1 for anomalies** and **1 for normal** points (a scikit-learn convention). We
convert that into a friendlier 0/1 flag (1 = anomaly) so it lines up with our `label` column.

In [ ]:
# TODO 1: create the model.
#   model = IsolationForest(contamination=0.1, random_state=42)

# TODO 2: fit the model and predict; store the -1/1 output in a new column.
#   df["pred"] = model.fit_predict(X)

# TODO 3: convert -1/1 into 1/0 (1 = anomaly) to match our label column.
#   df["pred_anomaly"] = (df["pred"] == -1).astype(int)

# Then look at how many anomalies it flagged:
#   df["pred_anomaly"].value_counts()

## Step 3 — Grade the model against our answer key

Now we bring back the `label` column we hid, and compare the model's guesses (`pred_anomaly`)
to the truth (`label`). Two standard tools:

- **Confusion matrix**: a 2×2 table of correct vs incorrect calls (true/false positives and
  negatives). It shows *how* the model is right or wrong.
- **Classification report**: turns that into familiar scores:
  - **precision** — of the readings it flagged, how many were truly incidents?
  - **recall** — of the true incidents, how many did it catch?
  - **f1-score** — a single number balancing the two.

For a safety system, **recall matters a lot** — missing a real oxygen emergency is far worse
than raising a false alarm.

In [ ]:
# TODO: import the metrics and print both the confusion matrix and the classification report.
#   from sklearn.metrics import classification_report, confusion_matrix
#   print(confusion_matrix(df["label"], df["pred_anomaly"]))
#   print(classification_report(df["label"], df["pred_anomaly"], digits=3))

## Step 4 — Did it catch the big one?

Scores are abstract; let's check the moment that matters. Filter to the readings where oxygen
actually fell below the safe line (19.5%) — the Day-5 scrubber fault — and see whether the
model flagged them.

In [ ]:
# TODO: show the low-oxygen rows and whether the model flagged them.
#   df[df.o2_pct < 19.5][["timestamp", "o2_pct", "co2_ppm", "label", "pred_anomaly"]]

### 🌱 Stretch (optional)

- **Rolling features.** How a value is *changing* can matter more than its current level. Try
  `df["o2_roll"] = df["o2_pct"].rolling(4).mean()` (a 1-hour moving average) as an extra
  feature and retrain.
- **Go supervised.** Train a `LogisticRegression` using the `label` column and compare its
  scores. When you *do* have labels, supervised models often win — but you need those labels.

> 🔗 **Looking ahead:** this detector is exactly the kind of tool ARIA will *call* in Module 5.
> A model that says "this reading is anomalous" becomes an action the assistant can take.

## ✅ Recap

You trained a model that learned "normal" from the data alone and flagged the unusual
readings — then you *measured* how well it did against a known answer key. That loop —
**train → predict → evaluate** — is the heartbeat of machine learning, and you'll see it
again and again.

# 1b · Train an anomaly model

**Core · Live-code · ~25 min**

Goal: teach a model to flag off-nominal readings **without** giving it the answers, then
check how well it found the known incidents.

We'll use scikit-learn's `IsolationForest` — an *unsupervised* anomaly detector. It learns
what "normal" looks like and flags the odd ones out.

> Fill in each `# TODO`. Solution: [`solution/01b_anomaly_model.ipynb`](solution/01b_anomaly_model.ipynb).

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../data/telemetry.csv", parse_dates=["timestamp"]).ffill().bfill()

# The sensor signals we'll learn from (NOT timestamp/module/label).
features = ["habitat_temp_c", "o2_pct", "co2_ppm", "power_kw", "battery_pct", "dust_index", "water_l"]
X = df[features]
X.head()

## Train the detector

`contamination` tells the model roughly what fraction of data is anomalous. Our known
incidents are about 10% of the week — a reasonable starting guess.

In [ ]:
# TODO: create an IsolationForest with contamination=0.1 and random_state=42
# model = ...

# TODO: fit it on X and predict. IsolationForest returns -1 for anomalies, 1 for normal.
# df["pred"] = model.fit_predict(X)

# TODO: convert to a 0/1 flag matching our label column (1 = anomaly)
# df["pred_anomaly"] = (df["pred"] == -1).astype(int)

## How well did it do?

We *do* have the ground-truth `label` column — use it to score the model.

In [ ]:
# TODO: import and print a classification_report comparing df["label"] to df["pred_anomaly"]
# from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# TODO: did it catch the big O2 drop? Look at the Day-5 afternoon rows.
# hint: df[(df.o2_pct < 19.5)][["timestamp", "o2_pct", "co2_ppm", "pred_anomaly"]]

### 🌱 Stretch

- Add a rolling feature, e.g. `df["o2_roll"] = df["o2_pct"].rolling(4).mean()`, and retrain.
- Try a **supervised** `LogisticRegression` using the `label` column and compare scores.
- This detector is exactly the kind of tool ARIA will call in Module 5!